# accuracy.py

This notebook is used to calculate the performance of a `sentence-transformers` model on a predefined set of validation examples. The validation set consists of 60 thousand enhanced and scrambled LOINC codes (enhancement means we have applied TtC synonym resolution, related-name insertion, word order permutation, and character deletion), 20 thousand each of long common name, short name, and display name. The file lives as a Data Asset in Azure Blob Storage and is read directly into local memory for computation.

It is broadly similar to the `Performance.py`, but is intended to evaluate whether any of the returned results have the same LOINC code, the same OID, or are part of the list of OIDs that are used to trigger a report for a condition in RCKMS.

While it is possible to run this notebook on CPU, we highly recommend using an appropriately powerful GPU (e.g. the NC24 A100 series) to speed up computation. The Approximate Nearest Neighbor search we use makes searching in memory almost instantaneous, but there is still a time-cost to encode each validation string input into a vector before semantic searching. Over 60 thousand encodings, this time adds up: GPU encoding is 10-15 times faster than CPU.


## Setup

Make sure that once the compute instance is running, you activate the kernel associated with the DIBBs Env in the upper right dropdown. Its packages are correctly optimized for this notebook and avoids some `numpy` instabilities plaguing Azure.

In [ ]:
pip install azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec hnswlib

Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [ ]:
# Authenticate to Key Vault
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential)

SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = 'workspaceblobstore'

_IMPORTANT_: This step can't be skipped, even though we're not directly using any of the `ml_client` functionality. This authentication and connection step allows us to use this notebook cleanly within our compute ecosystem. Basically, instantiating the class object acts as a connection that allows us to do everything that follows.

In [ ]:
from azure.ai.ml import MLClient

# Authenticate and connect to workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

Finally, we'll set the remainder of our imports and some global constants we'll be using.

The most important variables in this cell are the `MODEL VARIABLES` values, the `MODLE_NAME` and `EMBEDDING_SIZE`. The model name comes directly from the hugging face page for a particular model, and can be directly copied using the button beside the name on the web page. The embedding size for most models we work with is 768 (which is the industry standard dimensionality for any BERT-based transformer), but some are 1024 and a few are even higher. You can refer to the spreadsheet that tracks model performance for the exact size of a given model. If you set an embedding size here that proves to be incorrect later, when calculating the index, no harm will be done--the cell will simply halt and tell you the embedding dimensionality is wrong. Just come back here and change the value (likely to whichever of 1024 or 768 you haven't tried yet), then go back and pick up with indexing.

In [ ]:
import os
import random
import time
from typing import List

# MODEL VARIABLES
MODEL_NAME = "intfloat/e5-large-v2"
EMBEDDING_SIZE = 1024

# The name of the file in blob storage of the model's pickled vectors
SNOINC_CODES_FILE = "./loinc_lab_names_20251008.csv"
DATE = SNOINC_CODES_FILE.split("_")[-1].split(".")[0]
MODEL_FOLDER_NAME = f"loinc_lab_names_{MODEL_NAME.replace('/', '_')}_{DATE}"
MODEL_FOLDER = f"embeddings/refined/split/{MODEL_FOLDER_NAME}"
EMBEDDING_FILE = f"embeddings/loinc_lab_names_{MODEL_NAME.replace('/', '_')}_{DATE}"


# The name of the HNSW index file for this particular model
INDEX_FP = f"hnswlib_index_{MODEL_NAME.replace('/', '_')}_400.index"

# VALIDATION VARIABLES
VALIDATION_FILE = "embeddings/analysis/validation_set_60k_pairs.txt"
K_VALUES = [1, 3, 5, 10]

# Mapping files generated from build_evaluation_files.py in git repo
loinc_to_oids_file = "embeddings/analysis/loinc_to_oids.txt"
oid_to_conditions_file = "embeddings/analysis/oid_to_conditions.txt"

# Currently saving files locally; will update once code is fully vetted
JSONL_FP = f"./evaluation_results_{MODEL_NAME.replace('/', '_')}_{DATE}.jsonl"
ACCURACY_RESULTS = f"./accuracy_evaluation_results_{MODEL_NAME.replace('/', '_')}_{DATE}.jsonl"

**Important**: This cell determines whether the notebook will use Exact Neareset Neighbor search or Approximate Nearest Neighbor Search. Exact search is supported _only_ when the notebook is run using a GPU-enabled cluster, since the Tensor operations are valid only in a CUDA environment with `pytorch`. Further, exact search is computationally feasible only with a GPU providing a massive speed boost.

For TtC team purposes, we have found ANN using a compute instance with a lot of RAM and a large number of cores to be the most performant evaluation option. ANN is roughly twice as fast as Exact search, even with GPU-boosting, due to the speed of retrieval. GPU-boosted exact search is in turn ~10 times faster than exact search without GPU-boosting. For most use-cases, we advise using ANN.

In [ ]:
import torch

USE_EXACT_SEARCH = False

if USE_EXACT_SEARCH:
    assert torch.cuda.is_available()

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC container. Any `.txt` or `.csv` files need to be created as Data Assets (see sidebar on left), while embedding tensor files and any HNSW `.index` files do not, and can simply be loaded directly from storage.

In [ ]:
# Load up the validation set data
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)


## Step 2: Add loinc codes via snoinc files

In [ ]:
SNOINC_CODES_FILE = "./loinc_lab_names_20251008.csv"

# Load up the LOINC file data
print("Extracting SNOINC data to create name mapping...")

names_to_codes_map = {}

with fs.open(SNOINC_CODES_FILE) as fp:
    # First line is a header giving the column names
    lines_seen = 0
    for line in fp:
        if lines_seen == 0:
            lines_seen += 1
            continue
        
        # Azure Blob Storage is bytes-based, so we need to apply utf decoding
        # before we can use string operations
        line_str = line.decode("utf-8")
        if line_str.strip() != "":
            names = line_str.strip().split("|")
            # Skip lines that aren't real entries (formatting artifacts)
            if len(names) >= 4:
                loinc_code = names[0].strip()
                if loinc_code == "":
                    continue
                # Long Common Name
                if names[2].strip() != "":
                    names_to_codes_map[names[2].strip()] = loinc_code
                # Short Name
                if names[1].strip() != "":
                    names_to_codes_map[names[1].strip()] = loinc_code
                # Display Name
                if names[3].strip() != "":
                    names_to_codes_map[names[3].strip()] = loinc_code

# There should be ~240k loinc names
assert len(names_to_codes_map) > 0
print(f"{len(names_to_codes_map)} codes loaded")

## Step 3: Unpickle Embeddings

Using our mounted file system, we can directly open the embedding file and unpickle it. Remember, each embedding file is stored as a dictionary of not just the embeddings computed by the `sentence-transformers` model, but the standard LOINC codes associated with those embeddings. These are important later for measuring accuracy.

In [ ]:
import pickle

# Load up the pre-computed embeddings from the datastore
with fs.open(EMBEDDING_FILE) as fp:
    cache_data = pickle.load(fp)

name_codes = cache_data["codes"]
embeddings = cache_data["embeddings"]
loinc_type = cache_data["loinc_types"]
loinc_code = [names_to_codes_map[name] if name != "" else "" for name in name_codes]


if not USE_EXACT_SEARCH:
    # We move the embedded vectors to CPU for optimized searching later,
    # since the ANN indexer exists in CPU
    embeddings = embeddings.cpu().numpy()

## Step 4: Load HNSW Index

Whether computed from a previous Azure run, or computed locally and uploaded, we will use the embeddings to populate an HNSW index for fast Approximate Nearest Neighbor searching. The parameter values below govern the depth / connectivity of the search, but note that if the index was previously constructed, only the `EF_SEARCH` value will impact performance.

The `hnswlib` package _cannot_ directly open Azure binary files, which is how the FileSystemMount accesses and passes them. So what we need to do instead is first copy the file from the remote mount to local, working memory, and then we can access and open it. Once we've done that, it should be locally persisted for the remainder of our session.

During operation, this cell will create a temporary copy of the `.index` file in local, working memory. At the end of the cell, the file will be remote copied to Blob Storage and then deleted from local memory (on subsequent runs, it will simply be fetched from remote storage). You can verify the file has been cleaned by checking the sidebar to the left, under `Notebooks`.


In [ ]:
import hnswlib

# ANN INDEX VARIABLES
EF_CONSTRUCTION = 400
M_VALUE = 64
EF_SEARCH = 400

if not USE_EXACT_SEARCH:
    # Load up or create an index over the embedding data
    index = hnswlib.Index(space="cosine", dim=EMBEDDING_SIZE)

    # Azure will check blob storage first using the file mount
    print("Checking for cached ANN index...")
    if fs.exists("indexes/" + INDEX_FP):
        print("  Found cached index. Loading it...")

        # First, try to regularly load the index, in case we copied it here
        # from a previous run
        try:
            index.load_index(INDEX_FP)
        
        # If we can't open the file (because it's AzureML binary), then we
        # can create a local ported copy and open from that
        except:
            try:
                fs.get("indexes/" + INDEX_FP, '.')
                index.load_index(INDEX_FP)
            
            # If that doesn't work then the file is beyond the reach of mortal
            # hands and is best left undisturbed, like all sleeping gods
            except:
                print("Could not copy or load index")
        
    else:
        print("No locally cached index found. Creating hierarchical index...")
        index.init_index(
            max_elements=len(embeddings), ef_construction=EF_CONSTRUCTION, M=M_VALUE
        )
        index.add_items(embeddings, list(range(len(embeddings))))

        # Default is to save to local, working memory, so we'll need to remote copy
        # to Azure blob storage just like the reverse of copying from blob storage
        # Also clean up the local copy to avoid surplus memory charges
        index.save_index(INDEX_FP)
        fs.put(INDEX_FP, "/indexes/")
    os.remove(INDEX_FP)

    # The index should be holding approximately 276k embeddings so it better exceed 0
    assert index.get_current_count() > 0
    index.set_ef(EF_SEARCH)


## Step 5: Load Validation Set

With our file system mount, loading the validation set and preparing it for evaluation is straightforward. No need to worry about local copying for this data, Azure's `fs.open()` can simply parse the binary into a string codec for us.

In [ ]:
print("Loading validation set...")
examples = []
with fs.open(VALIDATION_FILE) as fp:
    for line in fp:
        # Blob storage is bytes-based, so we need to decode before string operations
        line_str = line.decode("utf-8")
        if line_str.strip() != "":
            examples.append(line_str.strip().split("|"))

# There are either 60k examples or 240k examples in this list, depending
# on whether the abridged set or full validation set is used
assert len(examples) >= 60000

## Step 6: Perform Evaluation

This cell carries out the trial run with the model and scores its performance on the validation data. It's largely a dictionary-based tracking function that accumulates some numbers into lists partitioned out by the K-value associated with the run. The search method of retrieving results is slightly different depending on whether exact search or ANN is used (i.e. slightly different unpacking of the `hits` list). 

When we use the `hnswlib` API to perform ANN, we get a pretty nested structure of a pair of lists denoting the search results and the _distances_ of those results to the input query. The only nuance to this function is unpacking those lists, converting distances into scores (since we want to measure similarity), and pairing up the found neighbor result with the standard LOINC code it represents, using our earlier unpickled Corpus ID indices.

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

import json
import time
import random

print("Instantiating language model...")
model = SentenceTransformer(MODEL_NAME)

print("Predicting and computing stats for validation set...")

random.shuffle(examples)

with open(JSONL_FP, "w", encoding="utf-8") as fp:
    for i, e in enumerate(examples):
        if i % 10_000 == 0:
            print(f"Calculated {i} of {len(examples)} examples.")

        expected_loinc = e[0].strip()
        expected_label = e[1].strip()
        query_input = e[2].strip()

        if USE_EXACT_SEARCH:
            enc = model.encode(query_input, convert_to_tensor=True)
        else:
            enc = model.encode(query_input)

        for k in K_VALUES:
            k_str = str(k)

            if USE_EXACT_SEARCH:
                raw_hits = util.semantic_search(enc, embeddings, top_k=k)[0]
                hits = [
                    {"corpus_id": int(h["corpus_id"]), "score": float(h["score"])}
                    for h in raw_hits
                ]
            else:
                embedding_ids, distances = index.knn_query(enc, k=k)
                hits = [
                    {"corpus_id": int(corpus_id), "score": float(1 - dist)}
                    for corpus_id, dist in zip(embedding_ids[0], distances[0])
                ]
                hits = sorted(hits, key=lambda x: x["score"], reverse=True)

            row = {
                "example_idx": i,
                "query_input": query_input,
                "expected_label": expected_label,
                "expected_loinc": expected_loinc,
                "k": k,
                "results": [
                    {
                        "rank": idx + 1,
                        "corpus_id": int(h["corpus_id"]),
                        "label": name_codes[int(h["corpus_id"])],
                        "loinc_type": loinc_type[int(h["corpus_id"])],
                        "loinc_code": loinc_code[int(h["corpus_id"])],
                        "score": float(h["score"]),
                    }
                    for idx, h in enumerate(hits)
                ],
            }
            fp.write(json.dumps(row, ensure_ascii=False) + "\n")


print("Wrote:")
print(" ", JSONL_FP)



## Step 7: Run evaluation paradigm

(Can skip re=running the calculation of examples if eval_results already exists; evaluation is by-far the longest step even on a powerful processor.)

- `oid_to_conditions.txt` is a json file that logs the OID-SNOMED condition ID key-pairs.
- `loinc_to_oids.txt` is a json file that logs the LOINC code to the array of 1+ OIDs that leverage that LOINC code.

Memo detailing how files are used is here: https://docs.google.com/document/d/1yA5NJ06mf1EfLZRmNrrNKopWL6ExMj-dPYKy8wlVDGs/edit?tab=t.0#heading=h.b1r0q3mit8hy

In [ ]:

import json

with fs.open(loinc_to_oids_file) as fp:
    loinc_to_oids = json.load(fp)

with fs.open(oid_to_conditions_file) as fp:
    oid_to_conditions = json.load(fp)

with open(JSONL_FP) as f:
    eval_data = [json.loads(line) for line in f if line.strip()]

status_priority = {
    "first-degree match": 5,
    "second-degree match, one unique condition": 4,
    "third-degree match, one unique condition": 3,
    "third-degree match, multiple unique conditions": 2,
    "no match": 1,
    "no OIDs returned, investigate LOINC validity": 0,
    "no conditions returned, investigate OID validity": 0,
    "no LOINC returned": 0,
}

for item in eval_data:
    expected_loinc = item.get("expected_loinc")
    for result in item.get("results"):
        returned_loinc = result.get("loinc_code")

        returned_oids = sorted(loinc_to_oids.get(returned_loinc, []) if returned_loinc else [])
        expected_oids = sorted(loinc_to_oids.get(expected_loinc, []) if expected_loinc else [])

        returned_conditions = sorted(
            list(set(oid_to_conditions.get(oid) for oid in returned_oids))
        )
        expected_conditions = sorted(
            list(set(oid_to_conditions.get(oid) for oid in expected_oids))
        )
        if returned_loinc is None:
            status = "no LOINC returned"
        elif returned_loinc == expected_loinc:
            status = "first-degree match"
        elif returned_oids == expected_oids and len(returned_conditions) == 1:
            status = "second-degree match, one unique condition"
        elif returned_conditions == expected_conditions and len(returned_conditions) == 1:
            status = "third-degree match, one unique condition"
        elif returned_conditions == expected_conditions and len(returned_conditions) > 1:
            status = "third-degree match, multiple unique conditions"
        elif not returned_oids:
            status = "no OIDs returned, investigate LOINC validity"
        elif not returned_conditions:
            status = "no conditions returned, investigate OID validity"
        else:
            status = "no match"
        result["status"] = status

        # determine best status for the overall item (i.e., did any of the results achieve a match)
        best_status = "no match"
        if status_priority.get(status, 0) > status_priority.get(best_status, 0):
            best_status = status

        if best_status == "first-degree match":
            break

    item["status"] = best_status


with open(ACCURACY_RESULTS, "w") as f:
    json.dump(eval_data, f, indent=2)
print("Wrote:")
print(" ", ACCURACY_RESULTS)